In [50]:
import pandas as pd

df1 = pd.read_csv("csvs/minhashblocksample_noblocks.ref.csv").rename(columns={"score": "noblocks_score"})
df2 = pd.read_csv("csvs/minhashblocksample_blocks.ref.csv").rename(columns={"score": "blocks_score"})
D = df1.merge(df2, on=["doc_id", "method"])
D["delta"] = D["blocks_score"] - D["noblocks_score"]

In [52]:
import pandas as pd

from scipy.stats import wilcoxon

from string import Template

for method in ["loss", "min_k", "ref-stablelm-base-alpha-3b-v2"]:

    stats = pd.read_csv("stats.csv.gz")

    method2pattern = {"loss": Template(f"csvs/minhashblocksample_$kind.lite.csv"),
                      "min_k": Template(f"csvs/minhashblocksample_$kind.lite.csv"),
                      "ref-stablelm-base-alpha-3b-v2": Template(f"csvs/minhashblocksample_$kind.ref.csv")}

    noblocks = pd.read_csv(method2pattern[method].substitute({'kind': "noblocks"})).rename(columns={"score": "noblocks_score"})
    blocks = pd.read_csv(method2pattern[method].substitute({'kind': "blocks"})).rename(columns={"score": "blocks_score"})

    noblocks = stats.merge(noblocks, left_on='url', right_on="doc_id").drop(columns=['size'])
    blocks = stats.merge(blocks, left_on='url', right_on="doc_id")

    D = noblocks.merge(blocks, on=["doc_id", "method"])
    D["delta"] = D["blocks_score"] - D["noblocks_score"]

    D = D[D["method"] == method]

    D["size_bin"] = pd.cut(D["size"], bins=range(0, 50, 10), right=True)

    df = D.groupby("size_bin", observed=True).agg(
        delta_mean=("delta", "mean"),
        count=("delta", "count")
    ).reset_index()

    df.to_csv(method + ".csv", index=False)
    stat, p = wilcoxon(D["delta"].to_list(), alternative="greater")
    print(f"{D['delta'].mean():.3g}", method, p)


0.0613 loss 2.617349818916068e-99
0.173 min_k 4.009449276298088e-68
0.065 ref-stablelm-base-alpha-3b-v2 5.690380484910952e-98
